In [1]:
from langchain_community.document_loaders import DirectoryLoader
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context, conditional
from ragas.testset.prompts import translate_all
from langchain_ollama import OllamaEmbeddings, ChatOllama

The output should be a well-formatted JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output JSON schema:
```
{"type": "object", "properties": {"answer": {"title": "Answer", "type": "string"}, "verdict": {"title": "Verdict", "type": "integer"}}, "required": ["answer", "verdict"]}
```

Do not return any preamble or explanations, return only a pure JSON string surrounded by triple backticks (```).


In [2]:
data = 'data'
language = "pt"
distributions = {
    simple:0.4,
    reasoning:0.2,
    multi_context:0.2,
    conditional:0.2
    }

cache_dir = './cache'
amount_of_tests = 64

In [3]:
translate_all(language, cache_dir)

In [4]:
loader = DirectoryLoader(data)
documents = loader.load()

for document in documents:
    document.metadata['filename'] = document.metadata['source']

In [5]:
model = ChatOllama(model='llama3.1')
embeddings = OllamaEmbeddings(model='llama3.1')

generator = TestsetGenerator.from_langchain(
    model,
    model,
    embeddings
)

In [6]:
testset = generator.generate_with_langchain_docs(documents, amount_of_tests, distributions, with_debugging_logs=True)

embedding nodes:   0%|          | 0/108 [00:00<?, ?it/s]

Generating:   0%|          | 0/65 [00:00<?, ?it/s]

[ragas.testset.filters.DEBUG] context scoring: {'clarity': 1, 'depth': 3, 'structure': 2, 'relevance': 3, 'score': 2.25}
[ragas.testset.evolutions.DEBUG] keyphrases in merged node: ['Temperatura Ambiente', 'Baterias', 'No-Break', 'Umidade Relativa', 'Altitude', 'Interface de Gerenciamento', 'Softwares de Gerenciamento', 'Ambientes e Sistemas Operacionais']

Failed to parse output. Returning None.


-------------Prompt-------------


Dado um contexto, execute a seguinte tarefa e produza a resposta no formato JSON VÁLIDO: Avalie o contexto fornecido e atribua uma pontuação numérica de 1 (baixo), 2 (médio) ou 3 (alto) para cada um dos seguintes critérios em sua resposta JSON:
"clareza": Avalie a precisão e a compreensibilidade das informações apresentadas. Pontuações altas (3) são reservadas para contextos que são precisos em suas informações e fáceis de entender. Pontuações baixas (1) são para contextos onde as informações são vagas ou difíceis de compreender.
"profundidade": Determine o

KeyboardInterrupt: 


Failed to parse output. Returning None.


-------------Prompt-------------


Dado um contexto, execute a seguinte tarefa e produza a resposta no formato JSON VÁLIDO: Avalie o contexto fornecido e atribua uma pontuação numérica de 1 (baixo), 2 (médio) ou 3 (alto) para cada um dos seguintes critérios em sua resposta JSON:
"clareza": Avalie a precisão e a compreensibilidade das informações apresentadas. Pontuações altas (3) são reservadas para contextos que são precisos em suas informações e fáceis de entender. Pontuações baixas (1) são para contextos onde as informações são vagas ou difíceis de compreender.
"profundidade": Determine o nível de exame detalhado e a inclusão de insights inovadores dentro do contexto. Uma pontuação alta indica uma análise abrangente e perspicaz, enquanto uma pontuação baixa sugere um tratamento superficial do tópico.
"estrutura": Avalie o quão bem o conteúdo está organizado e se ele flui logicamente. Pontuações altas são concedidas a contextos que demonstra

In [6]:
dataframe = testset.to_pandas()
dataframe.to_csv('testset.csv')

In [7]:
dataset = testset.to_dataset()
dataset.save_to_disk('testset')

Saving the dataset (0/1 shards):   0%|          | 0/3 [00:00<?, ? examples/s]